Install necessary libraries, if need

In [ ]:
# %pip install -r ../requirements.txt

Import necessary libraries

In [1]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import pandas as pd
from catboost import CatBoostClassifier
from rasterio.transform import from_origin

These are crops with the corresponding numbers

In [2]:
class_names = {
    '1': 'winter wheat',
    '2': 'spring oats',
    '3': 'spring barley',
    '4': 'spring rye',
    '5': 'corn',
    '6': 'soybean',
    '7': 'sunflower',
    '8': 'sugar beet',
    '9': 'rapeseed',
    '10': 'sorghum',
    '11': 'potato',
    '12': 'cotton',
    '13': 'spring wheat',
    '14': 'winter oats',
    '15': 'winter barley',
    '16': 'winter rye'
}

We consider an example of map generating using fine-tuned CropGRM-small model. Firslty, we should select features for CropGRM-small model from the dataset.

In [3]:
cols_to_select=  ['sum_t_4', 'sum_t_5', 'sum_t_6', 'sum_t_7', 'sum_t_8', 'sum_t_9', 'sum_t_10', 
                  'sum_prec_4', 'sum_prec_6', 'sum_prec_10', 'median_t_4', 'median_t_6', 'median_t_9', 
                  'median_t_10', 'ndre_S', 'median_red_fitted_8', 'median_nir_fitted_5', 'median_nir_fitted_8', 
                  'median_swir1_fitted_6', 'median_swir1_fitted_7', 'median_swir1_fitted_8', 'median_green_fitted_7', 
                  'median_green_fitted_8', 'median_swir2_fitted_5']

Set the path to the data

In [5]:
shp_path='../data/raw/fields.fgb'
preds_path='../data/processed/input_data_for_model.parquet.gzip'
raster_path='../data/final/CropMap_fields.tif'
output_shp_path='../data/final/CropMap_fields.fgb'

Load the model

In [6]:
model=CatBoostClassifier()
model.load_model('../models/finetuned_model.cbm')

Make predictions to your data

In [7]:
df_classes=pd.read_parquet(preds_path)
predictions=model.predict(df_classes[cols_to_select])
df_classes['class']=predictions

Сompare one dataframe with another by columns

In [8]:
gdf_polygons = gpd.read_file(shp_path)
gdf = gdf_polygons.merge(df_classes[['field_id', 'class']], on='field_id', how='left')
gdf['class'] = gdf['class'].fillna(0).astype(int)

If need, you can save the result as shapefile ESRI

In [9]:
gdf['class'] = gdf['class'].astype(str)
gdf['class_name']=gdf['class'].map(class_names)

gdf[['field_id', 'class_name', 'geometry']].to_file(output_shp_path, encoding='utf8')

Or you can save the result as raster

In [10]:
pixel_size = 10  
xmin, ymin, xmax, ymax = gdf.total_bounds
width = int((xmax - xmin) / pixel_size)
height = int((ymax - ymin) / pixel_size)
    
transform = from_origin(west=xmin, north=ymax, xsize=pixel_size, ysize=pixel_size)

shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf['class']))

raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    fill=0,  
    transform=transform,
    dtype='int32',
    all_touched=True
)

with rasterio.open(
    raster_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=raster.dtype,
    crs=gdf.crs,
    transform=transform,
) as dst:
    dst.write(raster, 1)